In [2]:
from pathlib import Path
import os
import glob
from functools import reduce


import pandas as pd
import numpy as np
from ipfn import ipfn

In [3]:
# Load constraints
CONSTRAINTS_DIR = Path("../data/constraints")
files = sorted(CONSTRAINTS_DIR.glob("*.csv"))
frames = []
for f in files:
    d = pd.read_csv(f)
    d = d.loc[:, ~d.columns.str.contains("^Unnamed")]
    frames.append(d.drop_duplicates(subset="area_code"))

cons = reduce(lambda l, r: pd.merge(l, r, on="area_code", how="outer"), frames)
cons.reset_index(inplace=True)
cons.info()

<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   index                          153 non-null    int64  
 1   area_code                      153 non-null    str    
 2   age_65_69                      153 non-null    float64
 3   age_70_74                      153 non-null    float64
 4   age_75_79                      153 non-null    float64
 5   age_80_84                      153 non-null    float64
 6   age_85_plus                    153 non-null    float64
 7   uk_born                        153 non-null    float64
 8   non_uk_born                    153 non-null    float64
 9   ethnicity_non_white            153 non-null    float64
 10  ethnicity_white                153 non-null    float64
 11  health_bad_health              153 non-null    float64
 12  health_fair_health             153 non-null    float64
 13  h

In [4]:
# Define constraint variables

# Define constraint variables
var_groups = {
    "partnership": ["partner_has_partner", "partner_does_not_have_partner"],
    "nssec":       ["nssec_managerial", "nssec_intermediate", "nssec_manual"],
    "health":      ["health_very_good_health", "health_good_health", "health_fair_health", "health_bad_health"],
    "age":         ["age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85_plus"],
    "ethnicity":   ["ethnicity_white", "ethnicity_non_white"],
    "sex":         ["male", "female"],
    "cob":         ["uk_born", "non_uk_born"],
}

constraint_vars = [v for vs in var_groups.values() for v in vs]

In [5]:
constraints_ipf = cons[["area_code", "area_name", "population_65plus", *constraint_vars]]

for col in constraint_vars:
    constraints_ipf[col] = (
        pd.to_numeric(constraints_ipf[col], errors="coerce") *
        pd.to_numeric(constraints_ipf["population_65plus"], errors="coerce")
    ).round()

constraints_ipf["population_65plus"] = (
    pd.to_numeric(constraints_ipf["population_65plus"], errors="coerce")
    .round()
    .astype(int)
)

constraints_ipf = constraints_ipf[
    ["area_code", "population_65plus"] + constraint_vars
].copy()

constraints_ipf.head()

,area_code,population_65plus,partner_has_partner,partner_does_not_have_partner,nssec_managerial,nssec_intermediate,nssec_manual,health_very_good_health,health_good_health,health_fair_health,...,age_70_74,age_75_79,age_80_84,age_85_plus,ethnicity_white,ethnicity_non_white,male,female,uk_born,non_uk_born
0,E06000001,18233,10135.0,8098.0,4724.0,3714.0,9796.0,2349.0,6458.0,6317.0,...,5084.0,3222.0,2463.0,2217.0,18086.0,147.0,8327.0,9906.0,17876.0,357.0
1,E06000002,24180,12770.0,11410.0,6531.0,5121.0,12528.0,3442.0,8895.0,7553.0,...,6464.0,4300.0,3236.0,2972.0,23182.0,998.0,10995.0,13185.0,22842.0,1338.0
2,E06000003,31747,18188.0,13559.0,9218.0,6673.0,15856.0,4587.0,12110.0,10177.0,...,8675.0,6341.0,4479.0,3755.0,31591.0,156.0,14610.0,17137.0,31168.0,579.0
3,E06000004,37184,21473.0,15711.0,11912.0,8329.0,16944.0,5757.0,14486.0,11589.0,...,10253.0,6872.0,4954.0,4312.0,36449.0,735.0,17180.0,20004.0,35933.0,1251.0
4,E06000005,22040,12386.0,9654.0,7208.0,5560.0,9272.0,3483.0,8853.0,6770.0,...,5980.0,4149.0,3121.0,2832.0,21708.0,332.0,10180.0,11860.0,21279.0,761.0


In [6]:
IMPUTED_ELSA_DIR = Path("../data/imputed_elsa")
files = sorted(IMPUTED_ELSA_DIR.glob("sms_imputed_m*.csv"))


all_groups = list(var_groups)

specifications = {
    "full_specification":       (all_groups, 20),
    "restricted_specification": ([g for g in all_groups if g not in ("cob", "ethnicity")], 20),
    "drop_health":              ([g for g in all_groups if g != "health"], 1),
    "drop_nssec":               ([g for g in all_groups if g != "nssec"], 1),
    "drop_partnership":         ([g for g in all_groups if g != "partnership"], 1),
    "drop_cob":                 ([g for g in all_groups if g != "cob"], 1),
    "drop_ethnicity":           ([g for g in all_groups if g != "ethnicity"], 1),
}

for spec, (groups_used, n_imp) in specifications.items():
    spec_vars = [v for g in groups_used for v in var_groups[g]]
    OUT_DIR = Path("../data/synthetic_populations") / spec
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    for m, f in enumerate(files[:n_imp], start=1):
        people = pd.read_csv(f)[["idauniq"] + spec_vars + ["has_any_care_needs"]]
        people[spec_vars] = people[spec_vars].astype(int)

        results = []
        for _, row in constraints_ipf.iterrows():
            df_area = people.copy()
            df_area["weight"] = 1.0

            targets, dims = [], []
            for v in spec_vars:
                targets.append(pd.Series({0: row["population_65plus"] - row[v], 1: row[v]}))
                dims.append([v])

            fit = ipfn.ipfn(df_area, targets, dims, weight_col="weight", verbose=1)
            df_area, converged = fit.iteration()

            print(f"{spec} | imputation {m} | area code {row['area_code']}")

            df_area["area"] = row["area_code"]
            results.append(df_area[["area", "idauniq", "weight", "has_any_care_needs"]])

        pd.concat(results, ignore_index=True).to_csv(OUT_DIR / f"simulation_m{m:02d}.csv", index=False)
        print(f"{spec} | simulation {m} saved")

ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000001
full_specification | imputation 1 | area code E06000002
full_specification | imputation 1 | area code E06000003
ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000004
full_specification | imputation 1 | area code E06000005
ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000006
full_specification | imputation 1 | area code E06000007
ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000008
ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000009
ipfn converged: convergence_rate not updating or below rate_tolerance
full_specification | imputation 1 | area code E06000010
full_specification |

KeyboardInterrupt: 

In [14]:
names = (pd.read_csv("../data/constraints/total.csv", usecols=["area_code", "area_name"])
         .drop_duplicates("area_code"))

for spec in ["full_specification", "restricted_specification"]:
    files = sorted(Path(f"../data/synthetic_populations/{spec}").glob("simulation_m*.csv"))

    est = {}
    for m, f in enumerate(files, start=1):
        sim = pd.read_csv(f)
        sim["w_care"] = sim["weight"] * sim["has_any_care_needs"]
        g = sim.groupby("area")[["w_care", "weight"]].sum()
        est[m] = 1000 * g["w_care"] / g["weight"]
    est = pd.DataFrame(est)          # rows = areas, columns = imputations

    # Rubin's rules (within-imputation variance = 0)
    m = est.shape[1]
    mean = est.mean(axis=1)
    sd = np.sqrt((1 + 1 / m) * est.var(axis=1))
    t = stats.t.ppf(0.975, m - 1)

    pooled = pd.DataFrame({
        "care_1000_est": mean,
        "care_1000_sd": sd,
        "care_1000_cv": 100 * sd / mean,
        "care_1000_lo": mean - t * sd,
        "care_1000_hi": mean + t * sd,
    })

    out = (constraints_ipf[["area_code", "population_65plus"]]
           .merge(pooled, left_on="area_code", right_index=True)
           .merge(names, on="area_code", how="left"))
    out.to_csv(f"../data/pooled_estimates/pooled_{spec}.csv", index=False)